# Lasso exploration: predicting USD/ZAR movement

$$\Delta USDZAR_{t+28} = USDZAR_{t+28} - USDZAR_t$$

In [9]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from src.testing_validation.model_test import calculate_rmse

In [10]:
combined_data = (
    pd.read_parquet("../data/train.parquet")
    .sort_values("date")
    .reset_index(drop=True)
)

combined_data["usd_zar_28_movement"] = (
    combined_data["usd_zar_28"] - combined_data["usd_zar"]
)
combined_data.head(1)

,date,gold_usd_per_oz,platinum_usd_per_oz,gold_usd_per_oz_return,platinum_usd_per_oz_return,us_fed_funds,us_5y_yield,vix,broad_usd_index,iron_ore_usd_per_tonne,...,usd_zar,usd_zar_28,usd_zar_1w_return,usd_zar_1m_return,usd_zar_3m_return,usd_zar_1m_volatility,richards_bay_coal_usd,sa_5y_cds_bp,sa_5y_yield,usd_zar_28_movement
0,2008-10-10,855.400024,996.700012,-0.019289,-0.007625,0.79,2.77,69.95,97.999,60.8,...,9.3626,10.0284,0.085265,0.066904,-0.004945,0.043376,112.4,455.4,9.115,0.6658


## Time-series-safe tuning and evaluation

Scaling is learned separately inside every fold. The 28-row gap reduces leakage from overlapping 28-day forecast horizons. The alpha grid spans strong to weak regularisation on a logarithmic scale.

In [11]:
ALPHA_GRID = np.logspace(-4, 0, 25)
FORECAST_GAP = 28


def make_lasso_search(n_splits=5):
    pipeline = Pipeline([
        ("scale", StandardScaler()),
        ("lasso", Lasso(max_iter=50_000)),
    ])
    inner_cv = TimeSeriesSplit(n_splits=n_splits, gap=FORECAST_GAP)
    return GridSearchCV(
        estimator=pipeline,
        param_grid={"lasso__alpha": ALPHA_GRID},
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
    )


def evaluate_movement_lasso(X, data, n_splits=5):
    """Tune on each training window and evaluate future USD/ZAR levels."""
    outer_cv = TimeSeriesSplit(n_splits=n_splits, gap=FORECAST_GAP)
    movement = data["usd_zar_28_movement"]
    fold_results = []

    for fold, (train_index, test_index) in enumerate(outer_cv.split(X)):
        search = make_lasso_search(n_splits=4)
        search.fit(X.iloc[train_index], movement.iloc[train_index])

        predicted_movement = search.predict(X.iloc[test_index])
        predicted_level = data["usd_zar"].iloc[test_index] + predicted_movement
        actual_level = data["usd_zar_28"].iloc[test_index]
        baseline_level = data["usd_zar"].iloc[test_index]

        model_rmse = root_mean_squared_error(actual_level, predicted_level)
        baseline_rmse = root_mean_squared_error(actual_level, baseline_level)
        fold_results.append({
            "fold": fold,
            "best_alpha": search.best_params_["lasso__alpha"],
            "model_rmse": model_rmse,
            "persistence_rmse": baseline_rmse,
            "improvement_vs_persistence": baseline_rmse - model_rmse,
        })

    results = pd.DataFrame(fold_results)
    print(results.to_string(index=False))
    print(f"\nMean model RMSE: {results['model_rmse'].mean():.4f}")
    print(f"Mean persistence RMSE: {results['persistence_rmse'].mean():.4f}")
    return results

## All available features

In [12]:
excluded_columns = ["date", "usd_zar_28", "usd_zar_28_movement"]
X_all = combined_data.drop(columns=excluded_columns)

assert X_all.select_dtypes(exclude="number").empty
all_feature_results = evaluate_movement_lasso(X_all, combined_data)

 fold  best_alpha  model_rmse  persistence_rmse  improvement_vs_persistence
    0    0.464159    0.349077          0.296887                   -0.052190
    1    0.215443    0.324753          0.315234                   -0.009519
    2    0.100000    0.652054          0.624738                   -0.027316
    3    0.100000    0.567606          0.567337                   -0.000269
    4    0.100000    0.781122          0.779810                   -0.001312

Mean model RMSE: 0.5349
Mean persistence RMSE: 0.5168


In [19]:
X_all.sa_cpi

0       45.0
1       45.0
2       45.0
3       45.0
4       45.0
        ... 
3185    81.3
3186    81.3
3187    81.3
3188    81.3
3189    81.3
Name: sa_cpi, Length: 3190, dtype: float64

## Economically engineered feature set

`sa_repo_rate - us_fed_funds` captures the policy-rate differential. Only 5-year yields are currently available, so the SA–US 5-year spread proxies for the requested 2-year spread. SA sovereign CDS captures country risk. GDP is excluded because its publication lag can cause look-ahead bias.

The data do not yet contain US inflation, so SA inflation is retained as a domestic control rather than mislabeled as an inflation differential. `broad_usd_index` is the available dollar-strength proxy, not ICE DXY.

In [24]:
engineered_data = combined_data.copy()
engineered_data["interest_rate_diff"] = (
    engineered_data["sa_repo_rate"] - engineered_data["us_fed_funds"]
)
engineered_data["sa_us_5y_yield_spread"] = (
    engineered_data["sa_5y_yield"] - engineered_data["us_5y_yield"]
)

#engineered_data["commodities"] = engineered_data.iron_ore_usd_per_tonne * engineered_data.gold_usd_per_oz * engineered_data.platinum_usd_per_oz * engineered_data.richards_bay_coal_usd

engineered_features = [
    "usd_zar",
    #"gold_usd_per_oz",
    #"platinum_usd_per_oz",
    #"richards_bay_coal_usd",
    #"iron_ore_usd_per_tonne",
    "brent_usd_per_barrel",
    "interest_rate_diff",
    "sa_us_5y_yield_spread",
    "sa_yoy_inflation",
    "sa_5y_cds_bp",
    "vix",
    "broad_usd_index",
    "sa_cpi",
    #"gold_usd_per_oz_return",
    #"platinum_usd_per_oz_return",
    "usd_zar_1w_return",
    "usd_zar_1m_return",
    "usd_zar_3m_return",
    "usd_zar_1m_volatility",
]

X_engineered = engineered_data[engineered_features]
assert X_engineered.select_dtypes(exclude="number").empty
engineered_results = evaluate_movement_lasso(X_engineered, engineered_data)

 fold  best_alpha  model_rmse  persistence_rmse  improvement_vs_persistence
    0    0.464159    0.349077          0.296887                   -0.052190
    1    0.100000    0.326000          0.315234                   -0.010765
    2    0.068129    0.712558          0.624738                   -0.087820
    3    0.100000    0.565882          0.567337                    0.001455
    4    0.100000    0.782100          0.779810                   -0.002289

Mean model RMSE: 0.5471
Mean persistence RMSE: 0.5168


## Fit deployable models on all training data

After unbiased outer-fold evaluation, tune once more using all available training data. These fitted searches contain both the scaler and the final Lasso model. Their predictions are movements and must be added to the current `usd_zar` value.

In [15]:
movement_target = combined_data["usd_zar_28_movement"]

final_all_model = make_lasso_search()
final_all_model.fit(X_all, movement_target)

final_engineered_model = make_lasso_search()
final_engineered_model.fit(X_engineered, movement_target)

print("All-feature alpha:", final_all_model.best_params_["lasso__alpha"])
print("Engineered-feature alpha:", final_engineered_model.best_params_["lasso__alpha"])


print(f"All model RMSE: {calculate_rmse(X_all, engineered_data["usd_zar_28_movement"], final_all_model)}")
print(f"Engineered model RMSE: {calculate_rmse(X_engineered, engineered_data["usd_zar_28_movement"], final_engineered_model)}")

# Example level prediction:
# future_usd_zar = current_usd_zar + final_engineered_model.predict(new_features)

All-feature alpha: 0.1
Engineered-feature alpha: 0.1
All model RMSE: 0.5227532455411268
Engineered model RMSE: 0.5227532455411268
